In [15]:
# 0) Import packages
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import HistGradientBoostingClassifier

# 1) Load data
TRAIN_PATH = "/Users/christinewu/Desktop/academic resources/Advanced Machine Learning/Homework Files/Homework1- Kaggle Competition/CSV Files/train.csv"
TEST_PATH = "/Users/christinewu/Desktop/academic resources/Advanced Machine Learning/Homework Files/Homework1- Kaggle Competition/CSV Files/test.csv"
SAMPLE_SUB_PATH = "/Users/christinewu/Desktop/academic resources/Advanced Machine Learning/Homework Files/Homework1- Kaggle Competition/CSV Files/sample_submission.csv"

TARGET = "Irrigation_Need"
ID_COL = "id"
RANDOM_STATE = 42
CV_FOLDS = 5
N_JOBS = -1

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

In [2]:
# 2) Split features / target
X = train_df.drop(columns=[TARGET])
y = train_df[TARGET].copy()
X = X.drop(columns=[ID_COL], errors="ignore")
X_test = test_df.drop(columns=[ID_COL], errors="ignore")

# Encode target
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [3]:
# 3) Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

In [4]:
# 4) Prepare data for HistGradientBoosting
#    - numeric: fill with median
#    - categorical: convert to integer category codes consistently
def prepare_hgb_data(X_train_raw: pd.DataFrame, X_test_raw: pd.DataFrame):
    X_train = X_train_raw.copy()
    X_test = X_test_raw.copy()

    # Fill numeric columns
    for col in numeric_features:
        med = X_train[col].median()
        X_train[col] = X_train[col].fillna(med)
        X_test[col] = X_test[col].fillna(med)

    # Encode categorical columns consistently across train/test
    for col in categorical_features:
        train_col = X_train[col].astype("string").fillna("MissingCategory")
        test_col = X_test[col].astype("string").fillna("MissingCategory")

        all_vals = pd.Index(train_col.unique()).union(pd.Index(test_col.unique()))
        mapping = {val: i for i, val in enumerate(all_vals)}

        X_train[col] = train_col.map(mapping).astype("int32")
        X_test[col] = test_col.map(mapping).astype("int32")

    return X_train, X_test

X_prepared, X_test_prepared = prepare_hgb_data(X, X_test)

categorical_indices = [X_prepared.columns.get_loc(col) for col in categorical_features]

In [5]:
# 5) HistGradientBoosting model + tuning
hgb = HistGradientBoostingClassifier(
    random_state=RANDOM_STATE,
    early_stopping=True,
    categorical_features=categorical_indices
)

param_dist = {
    "learning_rate": [0.03, 0.05, 0.08, 0.1],
    "max_iter": [150, 250, 350],
    "max_leaf_nodes": [15, 31, 63],
    "max_depth": [None, 6, 10],
    "min_samples_leaf": [20, 30, 50],
    "l2_regularization": [0.0, 0.1, 1.0],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    estimator=hgb,
    param_distributions=param_dist,
    n_iter=4,
    scoring="accuracy",
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True
)

In [6]:
# 6) Train
search.fit(X_prepared, y_encoded)

print("Best HistGradientBoosting params:")
print(search.best_params_)
print("Best CV accuracy:", round(search.best_score_, 5))

best_model = search.best_estimator_

Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best HistGradientBoosting params:
{'min_samples_leaf': 50, 'max_leaf_nodes': 31, 'max_iter': 350, 'max_depth': 6, 'learning_rate': 0.08, 'l2_regularization': 1.0}
Best CV accuracy: 0.98429


In [7]:
# 7) Fit on full train and predict test
best_model.fit(X_prepared, y_encoded)
test_pred_encoded = best_model.predict(X_test_prepared)
test_pred = label_encoder.inverse_transform(test_pred_encoded)

In [8]:
# 8) Save submission
submission = sample_sub.copy()
submission[TARGET] = test_pred
submission.to_csv("submission_boosting.csv", index=False)

## Additional Boosting Models and Faster Hyperparameter Testing

To expand my boosting analysis, I added XGBoost and CatBoost and tested lightweight hyperparameter settings. I reduced runtime by using 3-fold cross-validation, fewer boosting rounds, and shallower trees.

In [9]:
from sklearn.metrics import balanced_accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [10]:
# faster CV setup
cv_fast = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

### Preprocessing for XGBoost

In [11]:
numeric_transformer_xgb = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer_xgb = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

xgb_preprocessor = ColumnTransformer([
    ("num", numeric_transformer_xgb, numeric_features),
    ("cat", categorical_transformer_xgb, categorical_features)
])

### Prepare data for CatBoost

In [12]:
X_cat = X.copy()
X_test_cat = X_test.copy()

for col in categorical_features:
    X_cat[col] = X_cat[col].astype(str).fillna("Missing")
    X_test_cat[col] = X_test_cat[col].astype(str).fillna("Missing")

for col in numeric_features:
    median_val = X_cat[col].median()
    X_cat[col] = X_cat[col].fillna(median_val)
    X_test_cat[col] = X_test_cat[col].fillna(median_val)

cat_feature_idx = [X_cat.columns.get_loc(col) for col in categorical_features]

### Helper functions

In [13]:
def evaluate_xgb_model(model, X, y, preprocessor, cv):
    scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        X_train_fold = X.iloc[train_idx]
        X_valid_fold = X.iloc[valid_idx]
        y_train_fold = y[train_idx]
        y_valid_fold = y[valid_idx]

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train_fold, y_train_fold)
        preds = pipeline.predict(X_valid_fold)
        score = balanced_accuracy_score(y_valid_fold, preds)
        scores.append(score)

        print(f"Fold {fold} balanced accuracy: {score:.5f}")

    return np.mean(scores), np.std(scores)


def evaluate_cat_model(model, X, y, cat_features, cv):
    scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        X_train_fold = X.iloc[train_idx]
        X_valid_fold = X.iloc[valid_idx]
        y_train_fold = y[train_idx]
        y_valid_fold = y[valid_idx]

        model.fit(
            X_train_fold,
            y_train_fold,
            cat_features=cat_features,
            verbose=False
        )

        preds = model.predict(X_valid_fold)
        preds = np.array(preds).astype(int).reshape(-1)
        score = balanced_accuracy_score(y_valid_fold, preds)
        scores.append(score)

        print(f"Fold {fold} balanced accuracy: {score:.5f}")

    return np.mean(scores), np.std(scores)

### XGBoost: test 2 faster parameter sets

In [16]:
xgb_param_sets = {
    "XGB Set 1": {
        "n_estimators": 120,
        "max_depth": 4,
        "learning_rate": 0.08,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "min_child_weight": 2
    },
    "XGB Set 2": {
        "n_estimators": 160,
        "max_depth": 5,
        "learning_rate": 0.05,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "min_child_weight": 1
    }
}

xgb_results = []

for label, params in xgb_param_sets.items():
    print(f"\nEvaluating {label}")

    xgb_model = XGBClassifier(
        objective="multi:softmax",
        num_class=len(label_encoder.classes_),
        random_state=RANDOM_STATE,
        eval_metric="mlogloss",
        tree_method="hist",
        **params
    )

    mean_score, std_score = evaluate_xgb_model(
        xgb_model, X, y_encoded, xgb_preprocessor, cv_fast
    )

    xgb_results.append({
        "Model": "XGBoost",
        "Parameter Set": label,
        "CV Balanced Accuracy": round(mean_score, 5),
        "CV Std": round(std_score, 5)
    })

xgb_results_df = pd.DataFrame(xgb_results).sort_values(
    by="CV Balanced Accuracy", ascending=False
).reset_index(drop=True)

xgb_results_df


Evaluating XGB Set 1
Fold 1 balanced accuracy: 0.95591
Fold 2 balanced accuracy: 0.95746
Fold 3 balanced accuracy: 0.95921

Evaluating XGB Set 2
Fold 1 balanced accuracy: 0.95934
Fold 2 balanced accuracy: 0.95936
Fold 3 balanced accuracy: 0.96098


,Model,Parameter Set,CV Balanced Accuracy,CV Std
0,XGBoost,XGB Set 2,0.95989,0.00077
1,XGBoost,XGB Set 1,0.95753,0.00135


### CatBoost: test 2 faster parameter sets

In [17]:
cat_param_sets = {
    "CatBoost Set 1": {
        "iterations": 120,
        "depth": 5,
        "learning_rate": 0.08,
        "l2_leaf_reg": 3
    },
    "CatBoost Set 2": {
        "iterations": 160,
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 4
    }
}

cat_results = []

for label, params in cat_param_sets.items():
    print(f"\nEvaluating {label}")

    cat_model = CatBoostClassifier(
        loss_function="MultiClass",
        random_state=RANDOM_STATE,
        verbose=False,
        **params
    )

    mean_score, std_score = evaluate_cat_model(
        cat_model, X_cat, y_encoded, cat_feature_idx, cv_fast
    )

    cat_results.append({
        "Model": "CatBoost",
        "Parameter Set": label,
        "CV Balanced Accuracy": round(mean_score, 5),
        "CV Std": round(std_score, 5)
    })

cat_results_df = pd.DataFrame(cat_results).sort_values(
    by="CV Balanced Accuracy", ascending=False
).reset_index(drop=True)

cat_results_df


Evaluating CatBoost Set 1
Fold 1 balanced accuracy: 0.96064
Fold 2 balanced accuracy: 0.95930
Fold 3 balanced accuracy: 0.95854

Evaluating CatBoost Set 2
Fold 1 balanced accuracy: 0.96041
Fold 2 balanced accuracy: 0.95997
Fold 3 balanced accuracy: 0.95958


,Model,Parameter Set,CV Balanced Accuracy,CV Std
0,CatBoost,CatBoost Set 2,0.95998,0.00034
1,CatBoost,CatBoost Set 1,0.95949,0.00087


### Compare all boosting models

In [18]:
boosting_results = pd.concat([
    pd.DataFrame({
        "Model": ["HistGradientBoosting"],
        "Parameter Set": ["Best RandomizedSearchCV"],
        "CV Balanced Accuracy": [round(search.best_score_, 5)],
        "CV Std": [np.nan]
    }),
    xgb_results_df,
    cat_results_df
], ignore_index=True)

boosting_results = boosting_results.sort_values(
    by="CV Balanced Accuracy", ascending=False
).reset_index(drop=True)

boosting_results

,Model,Parameter Set,CV Balanced Accuracy,CV Std
0,HistGradientBoosting,Best RandomizedSearchCV,0.98429,NaN
1,CatBoost,CatBoost Set 2,0.95998,0.00034
2,XGBoost,XGB Set 2,0.95989,0.00077
3,CatBoost,CatBoost Set 1,0.95949,0.00087
4,XGBoost,XGB Set 1,0.95753,0.00135


### Train the best XGBoost model and save predictions

In [19]:
best_xgb_label = xgb_results_df.iloc[0]["Parameter Set"]
best_xgb_params = xgb_param_sets[best_xgb_label]

best_xgb_model = XGBClassifier(
    objective="multi:softmax",
    num_class=len(label_encoder.classes_),
    random_state=RANDOM_STATE,
    eval_metric="mlogloss",
    tree_method="hist",
    **best_xgb_params
)

best_xgb_pipeline = Pipeline([
    ("preprocessor", xgb_preprocessor),
    ("model", best_xgb_model)
])

best_xgb_pipeline.fit(X, y_encoded)
xgb_test_pred_encoded = best_xgb_pipeline.predict(X_test)
xgb_test_pred = label_encoder.inverse_transform(xgb_test_pred_encoded)

submission_xgb = sample_sub.copy()
submission_xgb[TARGET] = xgb_test_pred
submission_xgb.to_csv("submission_xgboost.csv", index=False)

submission_xgb.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


### Train the best CatBoost model and save predictions

In [20]:
best_cat_label = cat_results_df.iloc[0]["Parameter Set"]
best_cat_params = cat_param_sets[best_cat_label]

best_cat_model = CatBoostClassifier(
    loss_function="MultiClass",
    random_state=RANDOM_STATE,
    verbose=False,
    **best_cat_params
)

best_cat_model.fit(X_cat, y_encoded, cat_features=cat_feature_idx)

cat_test_pred_encoded = best_cat_model.predict(X_test_cat)
cat_test_pred_encoded = np.array(cat_test_pred_encoded).astype(int).reshape(-1)
cat_test_pred = label_encoder.inverse_transform(cat_test_pred_encoded)

submission_cat = sample_sub.copy()
submission_cat[TARGET] = cat_test_pred
submission_cat.to_csv("submission_catboost.csv", index=False)

submission_cat.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


### Boosting Results

In [21]:
boosting_results = pd.DataFrame({
    "Model": [
        "HistGradientBoosting",
        "CatBoost",
        "XGBoost",
        "CatBoost",
        "XGBoost"
    ],
    "Parameter Set": [
        "Best RandomizedSearchCV",
        "CatBoost Set 2",
        "XGB Set 2",
        "CatBoost Set 1",
        "XGB Set 1"
    ],
    "CV Balanced Accuracy": [
        0.98429,
        0.95998,
        0.95909,
        0.95949,
        0.95753
    ],
    "CV Std": [
        np.nan,
        0.00034,
        0.00077,
        0.00087,
        0.00135
    ],
    "Kaggle Score": [
        0.95923,
        0.95733,
        0.95624,
        np.nan,
        np.nan
    ]
})

boosting_results

,Model,Parameter Set,CV Balanced Accuracy,CV Std,Kaggle Score
0,HistGradientBoosting,Best RandomizedSearchCV,0.98429,NaN,0.95923
1,CatBoost,CatBoost Set 2,0.95998,0.00034,0.95733
2,XGBoost,XGB Set 2,0.95909,0.00077,0.95624
3,CatBoost,CatBoost Set 1,0.95949,0.00087,NaN
4,XGBoost,XGB Set 1,0.95753,0.00135,NaN


## Boosting Discussion

In this notebook, I compared three boosting approaches: HistGradientBoosting, XGBoost, and CatBoost. I also tested two different lightweight hyperparameter settings for both XGBoost and CatBoost so I could see whether changing depth, learning rate, and number of boosting rounds would change performance.

For XGBoost, Set 2 performed better than Set 1. XGBoost Set 2 had a cross-validation balanced accuracy of 0.95909, compared with 0.95753 for Set 1. Its Kaggle public score was 0.95624.

For CatBoost, Set 2 also performed better than Set 1. CatBoost Set 2 had the strongest CatBoost cross-validation score at 0.95998, compared with 0.95949 for Set 1. Its Kaggle public score was 0.95733.

Even though CatBoost slightly outperformed XGBoost, neither one beat my original HistGradientBoosting submission on the Kaggle leaderboard. HistGradientBoosting had the best Kaggle public score among the boosting models at 0.95923. This means that the newer boosting models were competitive, but their improvements were small and did not surpass my original boosting approach.

My main takeaway is that boosting model performance did change with different hyperparameter settings, so tuning mattered. However, the improvements between the models were fairly small, and the original HistGradientBoosting model remained the strongest boosting method in my work.

## Boosting Model Comparison

Comparing all of the boosting models, HistGradientBoosting performed best on the Kaggle leaderboard with a score of 0.95923. CatBoost was the next-best boosting model with a public score of 0.95733, followed by XGBoost at 0.95624.

Within the tuned models, CatBoost Set 2 performed slightly better than CatBoost Set 1, and XGBoost Set 2 performed slightly better than XGBoost Set 1. This shows that the hyperparameter changes did affect model performance, but only by a small amount.

Overall, the differences among the boosting models were not dramatic. The original HistGradientBoosting model remained the best boosting method, while CatBoost and XGBoost served as useful comparisons that helped show how different boosting algorithms and settings affect results.

### Leaderboard Results

In [22]:
leaderboard_results = pd.DataFrame({
    "Model": [
        "HistGradientBoosting",
        "Best CatBoost",
        "Best XGBoost"
    ],
    "Parameter Set": [
        "Best RandomizedSearchCV",
        "CatBoost Set 2",
        "XGB Set 2"
    ],
    "CV Balanced Accuracy": [
        0.98429,
        0.95998,
        0.95909
    ],
    "Kaggle Score": [
        0.95923,
        0.95733,
        0.95624
    ]
})

leaderboard_results

,Model,Parameter Set,CV Balanced Accuracy,Kaggle Score
0,HistGradientBoosting,Best RandomizedSearchCV,0.98429,0.95923
1,Best CatBoost,CatBoost Set 2,0.95998,0.95733
2,Best XGBoost,XGB Set 2,0.95909,0.95624
